# Chunk 06 — Model Architecture Verification (CPU-Only)

> **This notebook runs CPU-only.** Do not enable GPU accelerator in Settings for this chunk — it wastes GPU quota with no speed benefit, since the workload is a handful of tiny shape checks.

## 1. Install PyTorch Geometric (CPU-Only)

Since this notebook does not use a GPU, we only need the base `torch-geometric` package. The compiled CUDA extensions (`pyg-lib`, `torch-scatter`, `torch-sparse`) are optional GPU performance accelerators — `GATv2Conv` works correctly on CPU via PyG's built-in fallback implementations, so we skip them entirely.

In [9]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())  # Expected: False — that's fine

!pip install torch-geometric -q

device = torch.device('cpu')
print(f'\nUsing device: {device} (GPU intentionally not used for this chunk)')


Torch version: 2.10.0+cpu
CUDA available: False

Using device: cpu (GPU intentionally not used for this chunk)


## 2. Path Configuration (No Git Clone)

Set up local working directories without cloning the GitHub repo. This notebook is self-contained — it defines the model classes inline and only needs the attached Kaggle Datasets for sample graph construction.

In [10]:
import os

WORKING_DIR = '/kaggle/working'
LOCAL_SRC_DIR = f'{WORKING_DIR}/src'
os.makedirs(LOCAL_SRC_DIR, exist_ok=True)

RAW_CACHE_DIR = '/kaggle/input/datasets/naufelaurnob/antenna-raw-cache-all-grids'
SEED_MASK_DIR = '/kaggle/input/datasets/naufelaurnob/antenna-seed-masks'

# If the Kaggle-nested path convention applies instead, uncomment:
# RAW_CACHE_DIR = '/kaggle/input/datasets/naufelaurnob/antenna-raw-cache-all-grids'
# SEED_MASK_DIR = '/kaggle/input/datasets/naufelaurnob/antenna-seed-masks'

import torch_geometric
print(f'PyG version: {torch_geometric.__version__} | device: {device}')


PyG version: 2.8.0 | device: cpu


## 3. GATv2 Attention Block

A single GATv2 convolution layer wrapped with LayerNorm, a residual connection (with optional linear projection when input/output dimensions differ), and ReLU activation. Two of these form one "stage" of the network.

In [11]:
import torch
import torch.nn as nn
from torch_geometric.nn import GATv2Conv

class GATv2Block(nn.Module):
    def __init__(self, in_channels, out_channels, heads, edge_dim):
        super().__init__()
        self.conv = GATv2Conv(
            in_channels, out_channels // heads,
            heads=heads, edge_dim=edge_dim,
            concat=True, dropout=0.0
        )
        self.norm = nn.LayerNorm(out_channels)
        self.residual_proj = (nn.Linear(in_channels, out_channels)
                              if in_channels != out_channels else nn.Identity())
        self.act = nn.ReLU()

    def forward(self, x, edge_index, edge_attr):
        out = self.conv(x, edge_index, edge_attr=edge_attr)
        out = self.norm(out)
        out = self.act(out + self.residual_proj(x))
        return out


## 4. AntennaGNN Full Model

The complete GATv2-based surrogate model. Projects node and edge features, passes them through 4 blocks (8 GATv2 layers total), then combines a metal-only global mean pool with the virtual-node embedding to predict the 201-point S11 spectrum.

In [12]:
from torch_geometric.nn import global_mean_pool

class AntennaGNN(nn.Module):
    def __init__(self, node_feat_dim=5, edge_feat_dim=2,
                 hidden_dim=128, heads=8, edge_dim=16,
                 num_blocks=4, output_dim=201):
        super().__init__()
        self.input_proj = nn.Linear(node_feat_dim, hidden_dim)
        self.edge_proj  = nn.Linear(edge_feat_dim, edge_dim)

        self.blocks = nn.ModuleList()
        for i in range(num_blocks):
            self.blocks.append(nn.ModuleList([
                GATv2Block(hidden_dim, hidden_dim, heads, edge_dim),
                GATv2Block(hidden_dim, hidden_dim, heads, edge_dim),
            ]))

        self.readout_proj = nn.Linear(hidden_dim * 2, 256)
        self.output_mlp = nn.Sequential(
            nn.Linear(256, 512), nn.ReLU(),
            nn.LayerNorm(512),
            nn.Linear(512, output_dim)
        )

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        batch = data.batch

        x = self.input_proj(x)
        edge_attr = self.edge_proj(edge_attr)

        for block in self.blocks:
            for layer in block:
                x = layer(x, edge_index, edge_attr)

        # Metal-only pooling (metal = feature 0 of original input > 0.5)
        metal_mask = data.x[:, 0] > 0.5
        metal_x = x[metal_mask]
        metal_batch = batch[metal_mask]
        pooled = global_mean_pool(metal_x, metal_batch)  # (B, hidden_dim)

        # Virtual node embedding (is_seed == -1 flags the virtual node)
        virtual_mask = data.x[:, 3] == -1
        virtual_x = x[virtual_mask]  # (B, hidden_dim)

        combined = torch.cat([pooled, virtual_x], dim=-1)  # (B, 2*hidden_dim)
        out = self.readout_proj(combined)
        out = self.output_mlp(out)
        return out  # (B, 201)


## 5. Parameter Count Verification

Instantiate the model with default hyperparameters. The model stays on CPU by default — no `.to(device)` call needed. Verify the trainable parameter count falls within the expected range (800k–3M).

In [13]:
model = AntennaGNN()
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')
assert 500_000 <= total_params <= 3_000_000, f'Unexpected param count: {total_params}'
print('Parameter count OK ✓')


Total trainable parameters: 587,001
Parameter count OK ✓


## 6. Graph Construction Function

Define the `build_pyg_graph` function from the pyg-graph-construction skill. This converts a raw patch pattern and S11 spectrum into a PyG `Data` object with 5-feature nodes, 4-connectivity edges, and a virtual global node (flagged with `is_seed = -1`).

In [14]:
import numpy as np
from torch_geometric.data import Data

def build_pyg_graph(patch_pattern, s11_db, seed_mask, N):
    coords = np.argwhere(seed_mask)
    seed_r, seed_c = coords.mean(axis=0)

    node_feats = []
    for i in range(N):
        for j in range(N):
            metal    = float(patch_pattern[i, j])
            x_norm   = j / (N - 1)
            y_norm   = i / (N - 1)
            is_seed  = float(seed_mask[i, j])
            dist_f   = np.sqrt((i - seed_r)**2 + (j - seed_c)**2) / N
            node_feats.append([metal, x_norm, y_norm, is_seed, dist_f])

    # Virtual global node: is_seed = -1 flags it for model readout
    node_feats.append([0.0, 0.5, 0.5, -1.0, 0.0])
    node_feats = torch.tensor(node_feats, dtype=torch.float32)

    edge_src, edge_dst, edge_attr = [], [], []
    etype_map = {(1,1):0, (1,0):1, (0,1):2, (0,0):3}
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            m_ij = int(patch_pattern[i, j])
            for (di, dj, direction) in [(0,1,0),(0,-1,1),(-1,0,2),(1,0,3)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < N and 0 <= nj < N:
                    nidx = ni * N + nj
                    m_nb = int(patch_pattern[ni, nj])
                    etype = etype_map[(m_ij, m_nb)]
                    edge_src.append(idx); edge_dst.append(nidx)
                    edge_attr.append([etype, direction])

    global_idx = N * N
    for i in range(N):
        for j in range(N):
            if patch_pattern[i, j] == 1:
                idx = i * N + j
                edge_src += [global_idx, idx]
                edge_dst += [idx, global_idx]
                edge_attr += [[4, 4], [4, 4]]

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float32)
    y = torch.tensor(s11_db, dtype=torch.float32).unsqueeze(0)
    return Data(x=node_feats, edge_index=edge_index, edge_attr=edge_attr, y=y)


## 7. Build Sample Graphs from Raw Cache

Construct 3 test graphs (one each from the 25×25, 45×45, and 55×55 grids) directly from the raw `.pt` cache. Everything stays on CPU — no `.to()` calls needed.

In [15]:
samples = {}
for N in [25, 45, 55]:
    cache = torch.load(f'{RAW_CACHE_DIR}/raw_cache_{N}x{N}.pt', weights_only=False)
    seed_mask = np.load(f'{SEED_MASK_DIR}/seed_mask_{N}.npy')
    data = build_pyg_graph(cache['patterns'][0].numpy(),
                           cache['s11'][0].numpy(),
                           seed_mask, N)
    samples[N] = data
    print(f'Grid {N}x{N}: nodes={data.x.shape[0]}, edges={data.edge_index.shape[1]}')
    del cache


Grid 25x25: nodes=626, edges=3324
Grid 45x45: nodes=2026, edges=10500
Grid 55x55: nodes=3026, edges=14894


## 8. Forward Pass Shape Verification

Batch the 3 sample graphs together and run a forward pass. Also test each grid size individually with batch_size=1. This runs in seconds on CPU given the tiny batch size — no meaningful slowdown versus GPU for this verification task.

In [16]:
from torch_geometric.data import Batch

# Multi-graph batch test
batch = Batch.from_data_list([samples[25], samples[45], samples[55]])
model.eval()
with torch.no_grad():
    out = model(batch)
print(f'Batch output shape: {out.shape}')
assert out.shape == (3, 201), f'Expected (3, 201), got {out.shape}'

# Single-graph tests per grid size
for N in [25, 45, 55]:
    single = Batch.from_data_list([samples[N]])
    with torch.no_grad():
        out_single = model(single)
    assert out_single.shape == (1, 201), f'Grid {N}: expected (1, 201), got {out_single.shape}'
    print(f'Grid {N}x{N} single-graph: {out_single.shape} ✓')

print('\nAll shape checks PASSED (CPU) ✓')


Batch output shape: torch.Size([3, 201])
Grid 25x25 single-graph: torch.Size([1, 201]) ✓
Grid 45x45 single-graph: torch.Size([1, 201]) ✓
Grid 55x55 single-graph: torch.Size([1, 201]) ✓

All shape checks PASSED (CPU) ✓


## 9. Gradient Flow Check

Run one forward + backward pass and verify that every trainable parameter received a non-None gradient, confirming there are no dead branches in the computation graph.

In [17]:
model.train()
batch = Batch.from_data_list([samples[25], samples[45], samples[55]])

out = model(batch)
loss = out.sum()  # dummy loss
loss.backward()

dead_params = []
for name, p in model.named_parameters():
    if p.requires_grad and p.grad is None:
        dead_params.append(name)

if dead_params:
    print('WARNING: Parameters with no gradient:')
    for n in dead_params:
        print(f'  {n}')
    raise RuntimeError(f'{len(dead_params)} parameters have no gradient!')
else:
    print(f'Gradient flow: OK — all {total_params:,} parameters received gradients ✓')


Gradient flow: OK — all 587,001 parameters received gradients ✓


## 10. Write `model.py`

Save the verified `GATv2Block` and `AntennaGNN` classes to a standalone `model.py` file in the Kaggle working directory.

In [18]:
%%writefile /kaggle/working/src/model.py
"""
AntennaGNN — GATv2-based surrogate model for S11 spectrum prediction.

Architecture: 8 GATv2 layers (4 blocks x 2 layers), 8 attention heads,
128 hidden dim. Readout: metal-only global mean pool + virtual node
embedding -> concat -> MLP -> 201-dim S11 spectrum.
"""

import torch
import torch.nn as nn
from torch_geometric.nn import GATv2Conv, global_mean_pool


class GATv2Block(nn.Module):
    def __init__(self, in_channels, out_channels, heads, edge_dim):
        super().__init__()
        self.conv = GATv2Conv(
            in_channels, out_channels // heads,
            heads=heads, edge_dim=edge_dim,
            concat=True, dropout=0.0
        )
        self.norm = nn.LayerNorm(out_channels)
        self.residual_proj = (nn.Linear(in_channels, out_channels)
                              if in_channels != out_channels else nn.Identity())
        self.act = nn.ReLU()

    def forward(self, x, edge_index, edge_attr):
        out = self.conv(x, edge_index, edge_attr=edge_attr)
        out = self.norm(out)
        out = self.act(out + self.residual_proj(x))
        return out


class AntennaGNN(nn.Module):
    def __init__(self, node_feat_dim=5, edge_feat_dim=2,
                 hidden_dim=128, heads=8, edge_dim=16,
                 num_blocks=4, output_dim=201):
        super().__init__()
        self.input_proj = nn.Linear(node_feat_dim, hidden_dim)
        self.edge_proj  = nn.Linear(edge_feat_dim, edge_dim)

        self.blocks = nn.ModuleList()
        for i in range(num_blocks):
            self.blocks.append(nn.ModuleList([
                GATv2Block(hidden_dim, hidden_dim, heads, edge_dim),
                GATv2Block(hidden_dim, hidden_dim, heads, edge_dim),
            ]))

        self.readout_proj = nn.Linear(hidden_dim * 2, 256)
        self.output_mlp = nn.Sequential(
            nn.Linear(256, 512), nn.ReLU(),
            nn.LayerNorm(512),
            nn.Linear(512, output_dim)
        )

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        batch = data.batch

        x = self.input_proj(x)
        edge_attr = self.edge_proj(edge_attr)

        for block in self.blocks:
            for layer in block:
                x = layer(x, edge_index, edge_attr)

        # Metal-only pooling
        metal_mask = data.x[:, 0] > 0.5
        metal_x = x[metal_mask]
        metal_batch = batch[metal_mask]
        pooled = global_mean_pool(metal_x, metal_batch)

        # Virtual node embedding
        virtual_mask = data.x[:, 3] == -1
        virtual_x = x[virtual_mask]

        combined = torch.cat([pooled, virtual_x], dim=-1)
        out = self.readout_proj(combined)
        out = self.output_mlp(out)
        return out


Writing /kaggle/working/src/model.py


## 11. Display & Transfer Instructions

Print the saved `model.py` for easy copy-paste and provide instructions for transferring it to the GitHub repository.

In [19]:
!cat /kaggle/working/src/model.py

print('\n' + '='*70)
print('Copy the model.py content printed above (or download it directly')
print('from the Kaggle Output file browser at src/model.py) and save it')
print('to REPO_ROOT/src/model.py in your local repo via Antigravity,')
print('then commit and push to GitHub so Chunk 7 can import it with:')
print('  from model import AntennaGNN')
print()
print('This notebook intentionally did not attempt to git clone or push,')
print('since Chunk 4 showed the repo requires authentication that is not')
print('yet configured in this Kaggle environment.')
print('='*70)


"""
AntennaGNN — GATv2-based surrogate model for S11 spectrum prediction.

Architecture: 8 GATv2 layers (4 blocks x 2 layers), 8 attention heads,
128 hidden dim. Readout: metal-only global mean pool + virtual node
embedding -> concat -> MLP -> 201-dim S11 spectrum.
"""

import torch
import torch.nn as nn
from torch_geometric.nn import GATv2Conv, global_mean_pool


class GATv2Block(nn.Module):
    def __init__(self, in_channels, out_channels, heads, edge_dim):
        super().__init__()
        self.conv = GATv2Conv(
            in_channels, out_channels // heads,
            heads=heads, edge_dim=edge_dim,
            concat=True, dropout=0.0
        )
        self.norm = nn.LayerNorm(out_channels)
        self.residual_proj = (nn.Linear(in_channels, out_channels)
                              if in_channels != out_channels else nn.Identity())
        self.act = nn.ReLU()

    def forward(self, x, edge_index, edge_attr):
        out = self.conv(x, edge_index, edge_attr=edge_attr)
     